# Chapter 7: Relationships Between Variables
**Spine:** Think Stats by Allen Downey (adapted for applied analytics work)  
**Libraries:** Production-grade only — `numpy`, `pandas`, `scipy.stats`, `matplotlib`, `seaborn`

---
## Why this chapter matters

Understanding relationships between variables is the foundation of every analytical
question that matters in practice:

- Does income predict house value?
- Does tip amount grow with total bill?
- Does party size affect how much people tip?

This chapter builds the tools to answer those questions rigorously — and more importantly,
to know when an apparent relationship is real and when it is misleading.

**Topics covered:**
- Covariance — the raw measure of joint variation
- Pearson correlation — standardized linear relationship
- Spearman correlation — rank-based, robust to outliers and non-linearity
- Scatter plots and visual diagnosis
- Pearson vs Spearman — when to use which
- Correlation vs causation — the most important distinction in applied analytics
- Correlation matrix — relationships across many variables simultaneously

In [ ]:
import ssl
import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing

ssl._create_default_https_context = ssl.create_default_context(cafile=certifi.where())

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

housing = fetch_california_housing(as_frame=True).frame
tips    = sns.load_dataset('tips')

print('California Housing:', housing.shape)
print('Tips:              ', tips.shape)
print('California Housing columns:', list(housing.columns))
print('Tips columns:              ', list(tips.columns))

---
## 1. Covariance

$$\text{Cov}(X, Y) = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})$$

- **Positive** — when $X$ is above its mean, $Y$ tends to be above its mean
- **Negative** — when $X$ is above its mean, $Y$ tends to be below its mean
- **Zero** — no linear relationship

**Limitation:** magnitude depends on units — impossible to compare across variable pairs. Standardize into correlation.

In [ ]:
income    = housing['MedInc'].values
house_val = housing['MedHouseVal'].values

cov_manual = np.sum((income - income.mean()) * (house_val - house_val.mean())) / (len(income) - 1)
cov_matrix = np.cov(income, house_val)
cov_numpy  = cov_matrix[0, 1]

print('--- Covariance: MedInc vs MedHouseVal ---')
print(f'Manual:  {cov_manual:.4f}')
print(f'numpy:   {cov_numpy:.4f}')
print()
print('--- Covariance matrix ---')
print(f'  [Var(X),   Cov(X,Y)]   [{cov_matrix[0,0]:.4f}, {cov_matrix[0,1]:.4f}]')
print(f'  [Cov(Y,X), Var(Y)  ]   [{cov_matrix[1,0]:.4f}, {cov_matrix[1,1]:.4f}]')
print()
print('Diagonal = variance. Off-diagonal = covariance (symmetric).')

---
## 2. Pearson correlation

$$r = \frac{\text{Cov}(X, Y)}{\sigma_X \cdot \sigma_Y}$$

Constrained to $[-1, 1]$. Measures **linear** relationships only.

- $|r| > 0.7$ — strong
- $0.4 < |r| < 0.7$ — moderate
- $|r| < 0.4$ — weak

In [ ]:
r_manual         = cov_manual / (income.std(ddof=1) * house_val.std(ddof=1))
r_numpy          = np.corrcoef(income, house_val)[0, 1]
r_scipy, p_value = stats.pearsonr(income, house_val)

print('--- Pearson correlation: MedInc vs MedHouseVal ---')
print(f'Manual:  {r_manual:.4f}')
print(f'numpy:   {r_numpy:.4f}')
print(f'scipy:   {r_scipy:.4f}  (p-value: {p_value:.2e})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(income, house_val, alpha=0.05, color='steelblue', s=5)
axes[0].set_title(f'MedInc vs MedHouseVal (r = {r_scipy:.4f})')
axes[0].set_xlabel('Median income ($10,000s)')
axes[0].set_ylabel('Median house value ($100,000s)')

slope, intercept, r, p, se = stats.linregress(income, house_val)
x_line = np.linspace(income.min(), income.max(), 100)
axes[1].scatter(income, house_val, alpha=0.05, color='steelblue', s=5)
axes[1].plot(x_line, slope * x_line + intercept, color='crimson', linewidth=2,
             label=f'y = {slope:.3f}x + {intercept:.3f}')
axes[1].set_title('With regression line')
axes[1].set_xlabel('Median income ($10,000s)')
axes[1].set_ylabel('Median house value ($100,000s)')
axes[1].legend()

plt.tight_layout()
plt.savefig('pearson_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Spearman correlation
**Dataset:** Tips (n=244, real restaurant data)

Converts both variables to **ranks** first, then computes Pearson on the ranks.

$$r_s = 1 - \frac{6 \sum d_i^2}{n(n^2 - 1)}$$

Where $d_i$ is the rank difference between $x_i$ and $y_i$.

**Advantages over Pearson:**
- Robust to outliers
- Captures monotonic (not just linear) relationships
- Appropriate for ordinal variables

In [ ]:
total_bill = tips['total_bill'].values
tip        = tips['tip'].values

pearson_r,  pearson_p  = stats.pearsonr(total_bill, tip)
spearman_r, spearman_p = stats.spearmanr(total_bill, tip)

print('--- Pearson vs Spearman: total_bill vs tip ---')
print(f'Pearson r:   {pearson_r:.4f}  (p-value: {pearson_p:.4f})')
print(f'Spearman rs: {spearman_r:.4f}  (p-value: {spearman_p:.4f})')
print()
print('Both agree — the relationship is approximately linear.')
print('When they diverge significantly, the relationship is non-linear or outliers are influential.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(total_bill, tip, color='steelblue', alpha=0.6)
slope, intercept, _, _, _ = stats.linregress(total_bill, tip)
x_line = np.linspace(total_bill.min(), total_bill.max(), 100)
axes[0].plot(x_line, slope * x_line + intercept, color='crimson', linewidth=2)
axes[0].set_title(f'Raw values — Pearson r = {pearson_r:.4f}')
axes[0].set_xlabel('Total bill ($)')
axes[0].set_ylabel('Tip ($)')

bill_ranks = stats.rankdata(total_bill)
tip_ranks  = stats.rankdata(tip)
axes[1].scatter(bill_ranks, tip_ranks, color='darkorange', alpha=0.6)
slope_r, intercept_r, _, _, _ = stats.linregress(bill_ranks, tip_ranks)
x_rank = np.linspace(bill_ranks.min(), bill_ranks.max(), 100)
axes[1].plot(x_rank, slope_r * x_rank + intercept_r, color='crimson', linewidth=2)
axes[1].set_title(f'Ranked values — Spearman rs = {spearman_r:.4f}')
axes[1].set_xlabel('Rank of total bill')
axes[1].set_ylabel('Rank of tip')

plt.tight_layout()
plt.savefig('pearson_vs_spearman.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Pearson vs Spearman — when to use which

| Condition | Pearson | Spearman |
|---|---|---|
| Linear relationship | ✓ | ✓ |
| Monotonic but non-linear | ✗ | ✓ |
| Significant outliers | ✗ | ✓ |
| One variable is ordinal | ✗ | ✓ |
| Heavily skewed data | ✗ | ✓ |
| Both continuous, no outliers | ✓ | ✓ |

**Rule:** compute both. If they agree, Pearson is fine. If they diverge, prefer Spearman.

In [ ]:
# Effect of a single outlier
bill_with_outlier = np.append(total_bill, [200])
tip_with_outlier  = np.append(tip, [1])

pearson_clean,  _ = stats.pearsonr(total_bill, tip)
pearson_dirty,  _ = stats.pearsonr(bill_with_outlier, tip_with_outlier)
spearman_clean, _ = stats.spearmanr(total_bill, tip)
spearman_dirty, _ = stats.spearmanr(bill_with_outlier, tip_with_outlier)

print('--- Effect of a single outlier ---')
print(f'Pearson  — clean: {pearson_clean:.4f}  dirty: {pearson_dirty:.4f}  change: {pearson_dirty-pearson_clean:+.4f}')
print(f'Spearman — clean: {spearman_clean:.4f}  dirty: {spearman_dirty:.4f}  change: {spearman_dirty-spearman_clean:+.4f}')
print()
print('Pearson shifts significantly. Spearman barely moves.')

---
## 5. Correlation vs causation

Three possible explanations for any correlation between $X$ and $Y$:

1. $X$ causes $Y$
2. $Y$ causes $X$ (reverse causation)
3. $Z$ causes both (confounding)

**Establishing causation requires:** randomized controlled experiments or causal inference methods.  
Correlation is the start of an investigation, not the conclusion.

In [ ]:
# Latitude as a confounder for both income and house value
latitude = housing['Latitude'].values

r_lat_income, _ = stats.pearsonr(latitude, income)
r_lat_house,  _ = stats.pearsonr(latitude, house_val)
r_inc_house,  _ = stats.pearsonr(income, house_val)

print('--- Confounding variable: Latitude ---')
print(f'Latitude vs MedInc:      r = {r_lat_income:.4f}')
print(f'Latitude vs MedHouseVal: r = {r_lat_house:.4f}')
print(f'MedInc vs MedHouseVal:   r = {r_inc_house:.4f}')
print()
print('Latitude correlates with both income and house value.')
print('Part of the income-house value correlation may be explained by location')
print('rather than income directly causing higher house values.')

---
## 6. Correlation matrix

In [ ]:
corr_matrix = housing.corr(method='pearson')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Pearson correlation matrix — California Housing')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('--- Key correlations with MedHouseVal ---')
house_corr = corr_matrix['MedHouseVal'].drop('MedHouseVal').sort_values(ascending=False)
for var, r in house_corr.items():
    strength  = 'strong' if abs(r) > 0.5 else 'moderate' if abs(r) > 0.3 else 'weak'
    direction = 'positive' if r > 0 else 'negative'
    print(f'  {var:<15} r = {r:>6.3f}  ({strength} {direction})')

---
## 7. Utility functions — reusable in production

In [ ]:
def correlation_report(x: np.ndarray, y: np.ndarray,
                        x_label: str = 'X', y_label: str = 'Y') -> pd.DataFrame:
    """
    Compute both Pearson and Spearman correlation with p-values.
    Warns when they diverge significantly.
    """
    pearson_r,  pearson_p  = stats.pearsonr(x, y)
    spearman_r, spearman_p = stats.spearmanr(x, y)

    result = pd.DataFrame({
        'method':      ['Pearson', 'Spearman'],
        'correlation': [round(pearson_r,  4), round(spearman_r, 4)],
        'p_value':     [round(pearson_p,  4), round(spearman_p, 4)],
        'significant': [pearson_p < 0.05,     spearman_p < 0.05]
    })

    print(f'--- Correlation report: {x_label} vs {y_label} ---')
    print(result.to_string(index=False))

    divergence = abs(pearson_r - spearman_r)
    if divergence > 0.1:
        print(f'\nWarning: divergence = {divergence:.4f}. Prefer Spearman.')
    else:
        print(f'\nPearson and Spearman agree (divergence = {divergence:.4f}). Linear assumption reasonable.')

    return result


def scatter_with_correlation(x: np.ndarray, y: np.ndarray,
                              x_label: str = 'X', y_label: str = 'Y',
                              title: str = ''):
    """
    Scatter plot with Pearson and Spearman annotated.
    """
    r_p, _ = stats.pearsonr(x, y)
    r_s, _ = stats.spearmanr(x, y)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x, y, alpha=0.3, color='steelblue', s=10)

    slope, intercept, _, _, _ = stats.linregress(x, y)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, color='crimson', linewidth=2)

    ax.set_title(title or f'{x_label} vs {y_label}')
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.annotate(f'Pearson r = {r_p:.4f}\nSpearman rs = {r_s:.4f}',
                xy=(0.05, 0.88), xycoords='axes fraction', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    plt.tight_layout()
    return ax


correlation_report(income, house_val, 'MedInc', 'MedHouseVal')
print()
correlation_report(total_bill, tip, 'TotalBill', 'Tip')

---
## Summary

| Concept | Formula | Production tool |
|---|---|---|
| Covariance | $\frac{1}{n-1}\sum(x_i-\bar{x})(y_i-\bar{y})$ | `np.cov(x, y)` |
| Pearson $r$ | $\frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y}$ | `scipy.stats.pearsonr(x, y)` |
| Spearman $r_s$ | Pearson on ranks | `scipy.stats.spearmanr(x, y)` |
| Correlation matrix | All pairwise Pearson | `df.corr(method='pearson')` |

**Three rules for applied analytics work:**
1. Always visualize before computing — scatter plots reveal what coefficients hide
2. Always compute both Pearson and Spearman — divergence is a signal worth investigating
3. Correlation is never causation — it is the start of an investigation, not the conclusion

**Next:** Chapter 8 — Estimation